## ESA Anomaly Dataset 
### Data Processing and EDA

In [ ]:
%pip install pandas numpy matplotlib 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

### Dataset Paths

In [2]:
DATASET_PATH = Path("ESA-Mission1")

CHANNEL_PATH = DATASET_PATH / "channels"
TELECOMMAND_PATH = DATASET_PATH / "telecommands"

labels_path = DATASET_PATH / "labels.csv"
channels_meta_path = DATASET_PATH / "channels.csv"

### Load Metadata 

In [3]:
labels = pd.read_csv(labels_path)
channels_meta = pd.read_csv(channels_meta_path)

print("Labels shape:", labels.shape)
print("Channels metadata shape:", channels_meta.shape)

labels.head()

Labels shape: (3589, 4)
Channels metadata shape: (76, 5)


,ID,Channel,StartTime,EndTime
0,id_1,channel_12,2004-12-01T20:42:15.429Z,2004-12-08T22:55:45.429Z
1,id_1,channel_13,2004-12-01T20:42:15.429Z,2004-12-08T22:55:45.429Z
2,id_1,channel_14,2004-12-01T20:43:45.429Z,2004-12-02T02:57:15.429Z
3,id_1,channel_15,2004-12-01T20:45:00.429Z,2004-12-02T02:58:45.429Z
4,id_1,channel_16,2004-12-01T20:43:45.429Z,2004-12-16T16:52:30.429Z


### Identify Channels with Anomaly 

In [4]:
channels_with_anomalies = labels["Channel"].unique()

print("Number of anomalous channels:", len(channels_with_anomalies))
print(channels_with_anomalies[:10])

Number of anomalous channels: 58
<StringArray>
['channel_12', 'channel_13', 'channel_14', 'channel_15', 'channel_16',
 'channel_17', 'channel_18', 'channel_19', 'channel_20', 'channel_21']
Length: 10, dtype: str


### Select Channel Subset with Anomaly 

In [5]:
selected_channels = list(channels_with_anomalies[:20])
print("Selected channels:", selected_channels)

Selected channels: ['channel_12', 'channel_13', 'channel_14', 'channel_15', 'channel_16', 'channel_17', 'channel_18', 'channel_19', 'channel_20', 'channel_21', 'channel_22', 'channel_23', 'channel_24', 'channel_25', 'channel_26', 'channel_27', 'channel_28', 'channel_29', 'channel_30', 'channel_31']


### Load Channel Telemetry 

In [6]:
dfs = []

for ch in selected_channels:
    
    df = pd.read_pickle(CHANNEL_PATH / ch)
    
    df.rename(columns={ch: ch}, inplace=True)
    
    dfs.append(df)

dfs[0].head()

,channel_12
datetime,
2000-01-01 00:00:06.633,0.317175
2000-01-01 00:01:36.633,0.317175
2000-01-01 00:03:06.633,0.317175
2000-01-01 00:04:36.633,0.317175
2000-01-01 00:06:06.633,0.317175


### Merge Channels Into Multivariate Dataset

In [7]:
data = pd.concat(dfs, axis=1)

print(data.shape)

data.head()

(14258506, 20)


,channel_12,channel_13,channel_14,channel_15,channel_16,channel_17,channel_18,channel_19,channel_20,channel_21,channel_22,channel_23,channel_24,channel_25,channel_26,channel_27,channel_28,channel_29,channel_30,channel_31
datetime,,,,,,,,,,,,,,,,,,,,
2000-01-01 00:00:06.633,0.317175,0.371764,0.297205,0.130113,0.766769,0.349474,0.353997,0.293778,0.371764,0.304747,0.352119,0.136553,0.761253,0.289167,0.417318,0.279741,0.225152,0.306254,0.176514,0.352119
2000-01-01 00:01:36.633,0.317175,0.371764,0.297205,0.130113,0.766769,0.349474,0.353997,0.293778,0.370204,0.304747,0.352119,0.136553,0.761253,0.289167,0.415810,0.279741,0.225152,0.306254,0.155388,0.352119
2000-01-01 00:03:06.633,0.317175,0.359286,0.309268,0.130113,0.768608,0.349474,0.353997,0.292219,0.371764,0.304747,0.352119,0.136553,0.761253,0.289167,0.418826,0.271944,0.225152,0.306254,0.158409,0.350672
2000-01-01 00:04:36.633,0.317175,0.371764,0.297205,0.130113,0.766769,0.349474,0.353997,0.292219,0.370204,0.304747,0.352119,0.136553,0.761253,0.289167,0.417318,0.279741,0.225152,0.307761,0.176514,0.350672
2000-01-01 00:06:06.633,0.317175,0.370204,0.297205,0.130113,0.766769,0.349474,0.355505,0.292219,0.371764,0.304747,0.352119,0.136553,0.761253,0.286153,0.418826,0.279741,0.226712,0.307761,0.155388,0.356449


### Downsample the Data (Very Important)
#### Each channel has 10M+ rows, which is too large for ML

In [8]:
data = data.iloc[::100]

print("Downsampled shape:", data.shape)

Downsampled shape: (142586, 20)


### Convert Label Time Stamp

In [9]:
labels["StartTime"] = pd.to_datetime(labels["StartTime"])
labels["EndTime"] = pd.to_datetime(labels["EndTime"])

### Create Anomaly Label Column

In [10]:
data = data.copy()
data.index = pd.to_datetime(data.index, utc=True).tz_convert(None)

labels["StartTime"] = pd.to_datetime(labels["StartTime"], utc=True).dt.tz_convert(None)
labels["EndTime"] = pd.to_datetime(labels["EndTime"], utc=True).dt.tz_convert(None)

data["anomaly"] = 0

for _, row in labels.iterrows():
    ch = row["Channel"]
    
    if ch in selected_channels:
        start = row["StartTime"]
        end = row["EndTime"]
        
        mask = (data.index >= start) & (data.index <= end)
        data.loc[mask, "anomaly"] = 1

### View Dataset

In [11]:
print(data["anomaly"].value_counts())
data.head()

anomaly
0    126838
1     15748
Name: count, dtype: int64


,channel_12,channel_13,channel_14,channel_15,channel_16,channel_17,channel_18,channel_19,channel_20,channel_21,...,channel_23,channel_24,channel_25,channel_26,channel_27,channel_28,channel_29,channel_30,channel_31,anomaly
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 00:00:06.633,0.317175,0.371764,0.297205,0.130113,0.766769,0.349474,0.353997,0.293778,0.371764,0.304747,...,0.136553,0.761253,0.289167,0.417318,0.279741,0.225152,0.306254,0.176514,0.352119,0
2000-01-01 02:30:06.633,0.312495,0.371764,0.295691,0.130113,0.766769,0.352489,0.372088,0.284421,0.374883,0.303240,...,0.136553,0.761253,0.304243,0.417318,0.287541,0.234512,0.306254,0.158409,0.350672,0
2000-01-01 05:00:06.633,0.309375,0.374883,0.295691,0.132687,0.766769,0.353997,0.373596,0.290660,0.378001,0.304747,...,0.136553,0.761253,0.310275,0.411287,0.290660,0.237630,0.306254,0.155388,0.353560,0
2000-01-01 07:30:06.633,0.310935,0.376441,0.297205,0.131400,0.766769,0.360027,0.382643,0.309375,0.379561,0.304747,...,0.136553,0.761253,0.310275,0.408273,0.295338,0.228272,0.309268,0.155388,0.350672,0
2000-01-01 10:00:06.633,0.310935,0.376441,0.295691,0.132687,0.766769,0.363042,0.373596,0.318735,0.379561,0.304747,...,0.137840,0.761253,0.307259,0.403749,0.290660,0.218915,0.306254,0.155388,0.353560,0


#### Save the Preprocessed Dataset

In [12]:
data.to_csv("esa_anomaly.csv", index=True)
print("Saved:", Path("esa_anomaly.csv").resolve())

Saved: /Users/abdulhameed/Desktop/ESA Anomaly Dataset/esa_anomaly.csv
